In [0]:
%sql

CREATE OR REPLACE TEMP VIEW stg_res_partner AS
SELECT 
    customer_id,
    identification_type,
    identification_number,
    phone,
    email,
    created_at,
    updated_at,
    is_updated,
    is_deleted
FROM bakery_pos_develop.silver.res_partner
QUALIFY ROW_NUMBER() OVER(
    PARTITION BY customer_id
    ORDER BY updated_at DESC
) = 1

In [0]:
%sql

CREATE TABLE IF NOT EXISTS bakery_pos_develop.gold.dim_res_partner (
    customer_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    customer_id INT,
    identification_type STRING,
    identification_number STRING,
    phone STRING,
    email STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    is_updated BOOLEAN,
    is_deleted BOOLEAN,
    PRIMARY KEY(customer_sk) 
)

In [0]:
%sql
MERGE INTO bakery_pos_develop.gold.dim_res_partner AS tgt
USING stg_res_partner AS src
ON tgt.customer_id = src.customer_id

WHEN MATCHED AND src.updated_at > tgt.updated_at THEN
    UPDATE SET
        tgt.identification_type = src.identification_type,
        tgt.identification_number = src.identification_number,
        tgt.phone = src.phone,
        tgt.email = src.email,
        tgt.created_at = src.created_at,
        tgt.updated_at = src.updated_at,
        tgt.is_updated = src.is_updated,
        tgt.is_deleted = src.is_deleted

WHEN NOT MATCHED THEN
    INSERT (
        customer_id,
        identification_type,
        identification_number,
        phone,
        email,
        created_at,
        updated_at,
        is_updated,
        is_deleted
    )
    VALUES (
        src.customer_id,
        src.identification_type,
        src.identification_number,
        src.phone,
        src.email,
        src.created_at,
        src.updated_at,
        src.is_updated,
        src.is_deleted
    )